# Airport Flight Operations — Spark Fundamentals
Practicing PySpark DataFrame transformations, aggregations, and execution plan inspection on flight events data.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType,
)
from datetime import datetime

# inic local SparkSession
spark = SparkSession.builder.appName("flight_basics").master("local[*]").getOrCreate()

spark

26/11/05 19:46:08 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [ ]:
# sample flight departure events
flight_data = [
    (
        "IB3401",
        "Iberia",
        "MAD",
        "LHR",
        datetime(2026, 11, 5, 8, 0),
        datetime(2026, 11, 5, 8, 5),
    ),
    (
        "BA450",
        "British Airways",
        "LHR",
        "JFK",
        datetime(2026, 11, 5, 9, 30),
        datetime(2026, 11, 5, 10, 15),
    ),
    (
        "LH1102",
        "Lufthansa",
        "FRA",
        "MAD",
        datetime(2026, 11, 5, 11, 0),
        datetime(2026, 11, 5, 11, 0),
    ),
    (
        "IB3402",
        "Iberia",
        "MAD",
        "ORY",
        datetime(2026, 11, 5, 12, 15),
        datetime(2026, 11, 5, 13, 0),
    ),
    (
        "AF1024",
        "Air France",
        "CDG",
        "MAD",
        datetime(2026, 11, 5, 14, 0),
        datetime(2026, 11, 5, 14, 25),
    ),
    (
        "BA452",
        "British Airways",
        "LHR",
        "MAD",
        datetime(2026, 11, 5, 15, 0),
        None,
    ),  # Cancelled
    (
        "IB3405",
        "Iberia",
        "MAD",
        "BCN",
        datetime(2026, 11, 5, 16, 0),
        datetime(2026, 11, 5, 16, 45),
    ),
]

schema = StructType(
    [
        StructField("flight_id", StringType(), False),
        StructField("airline", StringType(), False),
        StructField("origin", StringType(), False),
        StructField("destination", StringType(), False),
        StructField("scheduled_dep", TimestampType(), False),
        StructField("actual_dep", TimestampType(), True),
    ]
)

df_flights = spark.createDataFrame(flight_data, schema=schema)
df_flights.show(truncate=False)

+---------+---------------+------+-----------+-------------------+-------------------+
|flight_id|airline        |origin|destination|scheduled_dep      |actual_dep         |
+---------+---------------+------+-----------+-------------------+-------------------+
|IB3401   |Iberia         |MAD   |LHR        |2026-11-05 08:00:00|2026-11-05 08:05:00|
|BA450    |British Airways|LHR   |JFK        |2026-11-05 09:30:00|2026-11-05 10:15:00|
|LH1102   |Lufthansa      |FRA   |MAD        |2026-11-05 11:00:00|2026-11-05 11:00:00|
|IB3402   |Iberia         |MAD   |ORY        |2026-11-05 12:15:00|2026-11-05 13:00:00|
|AF1024   |Air France     |CDG   |MAD        |2026-11-05 14:00:00|2026-11-05 14:25:00|
|BA452    |British Airways|LHR   |MAD        |2026-11-05 15:00:00|NULL               |
|IB3405   |Iberia         |MAD   |BCN        |2026-11-05 16:00:00|2026-11-05 16:45:00|
+---------+---------------+------+-----------+-------------------+-------------------+



### Task 1: Calculate Delays & Status
- Derive `delay_minutes`: difference in minutes between `actual_dep` and `scheduled_dep`.
- Derive `status`:
  - If `actual_dep` is null -> `"CANCELLED"`
  - If `delay_minutes` > 15 -> `"DELAYED"`
  - Otherwise -> `"ON_TIME"`

In [ ]:
df_flights_delayed = df_flights.withColumn(
    "delay_minutes",
    ((F.unix_timestamp("actual_dep") - F.unix_timestamp("scheduled_dep")) / 60).cast(
        "int"
    ),
).withColumn(
    "status",
    F.when(F.col("actual_dep").isNull(), "CANCELLED")
    .when(F.col("delay_minutes") > 15, "DELAYED")
    .otherwise("ON_TIME"),
)

df_flights_delayed.show()

+---------+---------------+------+-----------+-------------------+-------------------+-------------+---------+
|flight_id|        airline|origin|destination|      scheduled_dep|         actual_dep|delay_minutes|   status|
+---------+---------------+------+-----------+-------------------+-------------------+-------------+---------+
|   IB3401|         Iberia|   MAD|        LHR|2026-11-05 08:00:00|2026-11-05 08:05:00|            5|  ON_TIME|
|    BA450|British Airways|   LHR|        JFK|2026-11-05 09:30:00|2026-11-05 10:15:00|           45|  DELAYED|
|   LH1102|      Lufthansa|   FRA|        MAD|2026-11-05 11:00:00|2026-11-05 11:00:00|            0|  ON_TIME|
|   IB3402|         Iberia|   MAD|        ORY|2026-11-05 12:15:00|2026-11-05 13:00:00|           45|  DELAYED|
|   AF1024|     Air France|   CDG|        MAD|2026-11-05 14:00:00|2026-11-05 14:25:00|           25|  DELAYED|
|    BA452|British Airways|   LHR|        MAD|2026-11-05 15:00:00|               NULL|         NULL|CANCELLED|
|

### Task 2: Aggregate Delay Metrics by Airline
- Filter out cancelled flights.
- Group by `airline`.
- Compute:
  - `total_flights`: total non-cancelled flights
  - `delayed_flights`: count of flights where status is `"DELAYED"`
  - `avg_delay_min`: average delay in minutes (rounded to 1 decimal place)
- Order descending by `avg_delay_min`.

In [76]:
flights_not_cancelled = df_flights_delayed.filter(
    df_flights_delayed.status != "CANCELLED"
)  # narrow
flights_not_cancelled.groupBy(flights_not_cancelled.airline).agg(  # wide
    F.count("*").alias("total_flights"),
    F.count(F.when(F.col("status") == "DELAYED", 1).otherwise(0)).alias(
        "delayed_flights"
    ),
    F.round(F.avg("delay_minutes"), 1).alias("avg_delay_min"),
).orderBy(F.col("avg_delay_min").desc()).show()

+---------------+-------------+---------------+-------------+
|        airline|total_flights|delayed_flights|avg_delay_min|
+---------------+-------------+---------------+-------------+
|British Airways|            1|              1|         45.0|
|         Iberia|            3|              3|         31.7|
|     Air France|            1|              1|         25.0|
|      Lufthansa|            1|              1|          0.0|
+---------------+-------------+---------------+-------------+



In [77]:
# above answer is note correct, iberia has 2 delayed flights not 3 and lufthansa has 0 delayed flights
# F.count counts every row where the col is not null, .otherwise(0) is still being counted
# 2 options:

# we set otherwise as NULL so it is not counted
flights_not_cancelled = df_flights_delayed.filter(
    df_flights_delayed.status != "CANCELLED"
)  # narrow
flights_not_cancelled.groupBy(flights_not_cancelled.airline).agg(  # wide
    F.count("*").alias("total_flights"),
    F.count(F.when(F.col("status") == "DELAYED", 1)).alias(
        "delayed_flights"
    ),  # same as adding .otherwise(None), without otherwise sparks defaults to NULL
    F.round(F.avg("delay_minutes"), 1).alias("avg_delay_min"),
).orderBy(F.col("avg_delay_min").desc()).show()

# or we set sum as 0 if not delayed
flights_not_cancelled_stats = (
    flights_not_cancelled.groupBy(flights_not_cancelled.airline)
    .agg(  # wide
        F.count("*").alias("total_flights"),
        F.sum(F.when(F.col("status") == "DELAYED", 1).otherwise(0)).alias(
            "delayed_flights"
        ),
        F.round(F.avg("delay_minutes"), 1).alias("avg_delay_min"),
    )
    .orderBy(F.col("avg_delay_min").desc())
)
flights_not_cancelled_stats.show()

+---------------+-------------+---------------+-------------+
|        airline|total_flights|delayed_flights|avg_delay_min|
+---------------+-------------+---------------+-------------+
|British Airways|            1|              1|         45.0|
|         Iberia|            3|              2|         31.7|
|     Air France|            1|              1|         25.0|
|      Lufthansa|            1|              0|          0.0|
+---------------+-------------+---------------+-------------+

+---------------+-------------+---------------+-------------+
|        airline|total_flights|delayed_flights|avg_delay_min|
+---------------+-------------+---------------+-------------+
|British Airways|            1|              1|         45.0|
|         Iberia|            3|              2|         31.7|
|     Air France|            1|              1|         25.0|
|      Lufthansa|            1|              0|          0.0|
+---------------+-------------+---------------+-------------+



### Task 3: Execution Plan Inspection (`.explain()`)
- Call `.explain()` on your final DataFrame.
- Check where narrow operations (`Filter`, `Project`) run and where the shuffle (`Exchange`) occurs.

In [78]:
flights_not_cancelled_stats.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [avg_delay_min#1531 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(avg_delay_min#1531 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2292]
      +- HashAggregate(keys=[airline#26], functions=[count(1), sum(CASE WHEN (status#356 = DELAYED) THEN 1 ELSE 0 END), avg(delay_minutes#355)])
         +- Exchange hashpartitioning(airline#26, 200), ENSURE_REQUIREMENTS, [plan_id=2289]
            +- HashAggregate(keys=[airline#26], functions=[partial_count(1), partial_sum(CASE WHEN (status#356 = DELAYED) THEN 1 ELSE 0 END), partial_avg(delay_minutes#355)])
               +- Project [airline#26, delay_minutes#355, CASE WHEN isnull(actual_dep#30) THEN CANCELLED WHEN (delay_minutes#355 > 15) THEN DELAYED ELSE ON_TIME END AS status#356]
                  +- Project [airline#26, actual_dep#30, cast((cast((unix_timestamp(actual_dep#30, yyyy-MM-dd HH:mm:ss, Some(Europe/Madrid), true) - unix_timestamp(scheduled_dep#29, yyyy-

notice:
- explanation runs from most recent operation to the one written first
- each `Exchange` defines a different stage in the DAG.
- orderBy produces `rangepartitioning`
- groupBy produces `hashpartitioning`
- these lines correspond to the computation of avg_delay-minutes.
```
      +- HashAggregate(keys=[airline#26], functions=[count(1), sum(CASE WHEN (status#356 = DELAYED) THEN 1 ELSE 0 END), avg(delay_minutes#355)])
         +- Exchange hashpartitioning(airline#26, 200), ENSURE_REQUIREMENTS, [plan_id=1885]
            +- HashAggregate(keys=[airline#26], functions=[partial_count(1), partial_sum(CASE WHEN (status#356 = DELAYED) THEN 1 ELSE 0 END), partial_avg(delay_minutes#355)])
```
   - the first hashaggregate is a partial aggregation (map side combine): to shrink network traffic, spark runs the function within the partion and holds it locally as a partial_sum
   - the exchange hashpartitioning is the actual shuffle, the stage boundary. spark uses the airline so all rows for the airline end up in the same executor. (200 is the def number of spark partitions)
   - the second (top) hashaggregate is the reduce and final merge. the downstream executor takes all the partial_sums and counts for that airline and computes the final    
  avg_delay_min and total counts.



### Task 4: String Formatting, Derived Metrics & SQL "HAVING" Equivalent
1. In `df_flights_delayed`:
   - Derive `route`: formatted string as `"MAD -> LHR"` using `origin` and `destination`.
   - Derive `airline_code`: extract the 2-letter prefix from `flight_id` (e.g. `"IB"`, `"BA"`).
2. In `flights_not_cancelled_stats`:
   - Add `delayed_pct`: `(delayed_flights / total_flights) * 100`, rounded to 1 decimal place.
   - Filter to keep only airlines with at least 2 total flights (the PySpark equivalent to SQL `HAVING total_flights >= 2`).

In [ ]:
flights_delayed_enriched = df_flights_delayed.withColumn(
    "route",
    F.concat(F.col("origin"), F.lit("→"), F.col("destination")),
).withColumn("airline_code", F.upper(F.substring("airline", pos=0, len=2)))

flights_delayed_enriched.show()

+---------+---------------+------+-----------+-------------------+-------------------+-------------+---------+-------+------------+
|flight_id|        airline|origin|destination|      scheduled_dep|         actual_dep|delay_minutes|   status|  route|airline_code|
+---------+---------------+------+-----------+-------------------+-------------------+-------------+---------+-------+------------+
|   IB3401|         Iberia|   MAD|        LHR|2026-11-05 08:00:00|2026-11-05 08:05:00|            5|  ON_TIME|MAD→LHR|          IB|
|    BA450|British Airways|   LHR|        JFK|2026-11-05 09:30:00|2026-11-05 10:15:00|           45|  DELAYED|LHR→JFK|          BR|
|   LH1102|      Lufthansa|   FRA|        MAD|2026-11-05 11:00:00|2026-11-05 11:00:00|            0|  ON_TIME|FRA→MAD|          LU|
|   IB3402|         Iberia|   MAD|        ORY|2026-11-05 12:15:00|2026-11-05 13:00:00|           45|  DELAYED|MAD→ORY|          IB|
|   AF1024|     Air France|   CDG|        MAD|2026-11-05 14:00:00|2026-11-05

In [ ]:
flights_not_cancelled_stats.withColumn(
    "delayed_pct",
    F.round(F.col("delayed_flights") / F.col("total_flights") * 100, scale=1),
).show()

+---------------+-------------+---------------+-------------+-----------+
|        airline|total_flights|delayed_flights|avg_delay_min|delayed_pct|
+---------------+-------------+---------------+-------------+-----------+
|British Airways|            1|              1|         45.0|      100.0|
|         Iberia|            3|              2|         31.7|       66.7|
|     Air France|            1|              1|         25.0|      100.0|
|      Lufthansa|            1|              0|          0.0|        0.0|
+---------------+-------------+---------------+-------------+-----------+



### Task 5: Compound Boolean Filters, Null Handling & Deduplication
A raw batch of flight radar updates arrives with dirty data:
- Duplicate flight records
- Missing/null airline names

**Tasks:**
1. Replace null `airline` with `"UNKNOWN"`.
2. Filter for flights where: (`origin == "MAD"` AND `delay_minutes > 15`) OR (`destination == "JFK"`).
   *(Remember: In PySpark, bitwise `&` / `|` require parentheses around each condition!)*
3. Deduplicate by `flight_id` using `.dropDuplicates()`.

In [ ]:
# Dirty batch of flight radar updates
dirty_flight_data = [
    ("IB3401", "Iberia", "MAD", "LHR", 5),
    ("IB3401", "Iberia", "MAD", "LHR", 5),  # Exact duplicate
    ("BA450", None, "LHR", "JFK", 45),  # Missing airline, JFK destination
    ("LH1102", "Lufthansa", "FRA", "MAD", 0),
    ("IB3402", "Iberia", "MAD", "ORY", 45),  # MAD origin & delay > 15
    ("IB3402", None, "MAD", "ORY", 50),  # Duplicate flight_id with conflicting info
    ("AF1024", "Air France", "CDG", "MAD", 25),
]

df_dirty = spark.createDataFrame(
    dirty_flight_data,
    ["flight_id", "airline", "origin", "destination", "delay_minutes"],
)
df_dirty.show()


+---------+----------+------+-----------+-------------+
|flight_id|   airline|origin|destination|delay_minutes|
+---------+----------+------+-----------+-------------+
|   IB3401|    Iberia|   MAD|        LHR|            5|
|   IB3401|    Iberia|   MAD|        LHR|            5|
|    BA450|      NULL|   LHR|        JFK|           45|
|   LH1102| Lufthansa|   FRA|        MAD|            0|
|   IB3402|    Iberia|   MAD|        ORY|           45|
|   IB3402|      NULL|   MAD|        ORY|           50|
|   AF1024|Air France|   CDG|        MAD|           25|
+---------+----------+------+-----------+-------------+



In [ ]:
# find duplicates
total_dirty = df_dirty.count()
nodups = df_dirty.drop_duplicates(subset=["flight_id"]).count()  # triggers a shuffle
print(
    "are there duplicated fligths: ",
    total_dirty > df_dirty.select("flight_id").distinct().count(),
)
print("how many duplicates within the new data: ", total_dirty - nodups)

# duplicated flight
print("exact duplicated flights: ")
df_dirty.groupBy(df_dirty.columns).count().filter(
    F.col("count") > 1
).show()  # triggers a shuffle

print("flight_id duplicated flights: ")
df_dirty.groupBy("flight_id").count().filter(F.col("count") > 1).show()

df_dirty_quick_clean = df_dirty.dropDuplicates(
    subset=["flight_id"]
)  ## how can we asure we keep the first occurrnece
df_dirty_quick_clean.show()

are there duplicated fligths:  True
how many duplicates within the new data:  2
exact duplicated flights: 
+---------+-------+------+-----------+-------------+-----+
|flight_id|airline|origin|destination|delay_minutes|count|
+---------+-------+------+-----------+-------------+-----+
|   IB3401| Iberia|   MAD|        LHR|            5|    2|
+---------+-------+------+-----------+-------------+-----+

flight_id duplicated flights: 
+---------+-----+
|flight_id|count|
+---------+-----+
|   IB3401|    2|
|   IB3402|    2|
+---------+-----+

+---------+----------+------+-----------+-------------+
|flight_id|   airline|origin|destination|delay_minutes|
+---------+----------+------+-----------+-------------+
|   AF1024|Air France|   CDG|        MAD|           25|
|    BA450|      NULL|   LHR|        JFK|           45|
|   IB3401|    Iberia|   MAD|        LHR|            5|
|   IB3402|    Iberia|   MAD|        ORY|           45|
|   LH1102| Lufthansa|   FRA|        MAD|            0|
+--------

In [ ]:
df_dirty.fillna(value="UNKNOWN", subset=["airline"]).filter(
    ((F.col("origin") == "MAD") & (F.col("delay_minutes") > 15))
    | (F.col("destination") == "JFK")
).dropDuplicates(subset=["flight_id"]).show()

+---------+-------+------+-----------+-------------+
|flight_id|airline|origin|destination|delay_minutes|
+---------+-------+------+-----------+-------------+
|    BA450|UNKNOWN|   LHR|        JFK|           45|
|   IB3402| Iberia|   MAD|        ORY|           45|
+---------+-------+------+-----------+-------------+



### Task 6: Pivoting / Cross-Tabulation (Airport vs Airline)
*How many flights did each airline operate out of each origin airport?*

- Group by `origin`.
- Pivot on `airline`.
- Count the flights and replace `NULL` counts with `0`.
- *Concept check:* Why should `.pivot()` be used cautiously on high-cardinality columns?

In [117]:
relevant_columns = ["airline_code", "airline", "origin", "flight_id"]
subset_flights = flights_delayed_enriched.select(relevant_columns)
subset_flights.show()

+------------+---------------+------+---------+
|airline_code|        airline|origin|flight_id|
+------------+---------------+------+---------+
|          IB|         Iberia|   MAD|   IB3401|
|          BR|British Airways|   LHR|    BA450|
|          LU|      Lufthansa|   FRA|   LH1102|
|          IB|         Iberia|   MAD|   IB3402|
|          AI|     Air France|   CDG|   AF1024|
|          BR|British Airways|   LHR|    BA452|
|          IB|         Iberia|   MAD|   IB3405|
+------------+---------------+------+---------+



In [ ]:
# how many flights did each airline operate out of each origin airport
# flights per airline per airport, the count is affected by the origin airport
# this produces a tall or long table: scalable and fixed schema (no matter if a new airline suddenly has flights to these origins for example)
subset_flights.groupBy("origin", "airline").count().show()

+------+---------------+-----+
|origin|        airline|count|
+------+---------------+-----+
|   MAD|         Iberia|    3|
|   LHR|British Airways|    2|
|   FRA|      Lufthansa|    1|
|   CDG|     Air France|    1|
+------+---------------+-----+



In [ ]:
# how many flights did each airline operate out of each origin airport
# flights per airline per airport, the count is affected by the origin airport 
# wide format, better human readability
# the cost of a pivot: spark needs to know the schema of a df before the transformation starts, 
#   when we do a pivot, spark adds a job to scan the airlines to know the columns it'll need to create (distinct values of the col to pivot)
wide_subset_flights = subset_flights.groupBy("origin").pivot("airline").count().fillna(0)
wide_subset_flights.show()

+------+----------+---------------+------+---------+
|origin|Air France|British Airways|Iberia|Lufthansa|
+------+----------+---------------+------+---------+
|   FRA|         0|              0|     0|        1|
|   CDG|         1|              0|     0|        0|
|   LHR|         0|              2|     0|        0|
|   MAD|         0|              0|     3|        0|
+------+----------+---------------+------+---------+



In [129]:
wide_subset_flights.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [origin#27, coalesce(__pivot_count(1) AS count AS `count(1) AS count`#3295[0], 0) AS Air France#3308L, coalesce(__pivot_count(1) AS count AS `count(1) AS count`#3295[1], 0) AS British Airways#3309L, coalesce(__pivot_count(1) AS count AS `count(1) AS count`#3295[2], 0) AS Iberia#3310L, coalesce(__pivot_count(1) AS count AS `count(1) AS count`#3295[3], 0) AS Lufthansa#3311L]
   +- HashAggregate(keys=[origin#27], functions=[pivotfirst(airline#26, count(1) AS count#3285L, Air France, British Airways, Iberia, Lufthansa, 0, 0)])
      +- Exchange hashpartitioning(origin#27, 200), ENSURE_REQUIREMENTS, [plan_id=7709]
         +- HashAggregate(keys=[origin#27], functions=[partial_pivotfirst(airline#26, count(1) AS count#3285L, Air France, British Airways, Iberia, Lufthansa, 0, 0)])
            +- HashAggregate(keys=[origin#27, airline#26], functions=[count(1)])
               +- Exchange hashpartitioning(origin#27, airline#26, 2

In [130]:
# how many flights did each airline operate out of each origin airport
# flights per airline per airport, the count is affected by the origin airport 
# wide format, better human readability
# to optimize the cost of a pivot: 
known_airlines = ["Iberia", "British Airways", "Lufthansa", "Air France"]
wide_subset_flights_opt = subset_flights.groupBy("origin").pivot("airline", values=known_airlines).count().fillna(0)
wide_subset_flights_opt.show()
wide_subset_flights_opt.explain()

+------+------+---------------+---------+----------+
|origin|Iberia|British Airways|Lufthansa|Air France|
+------+------+---------------+---------+----------+
|   FRA|     0|              0|        1|         0|
|   CDG|     0|              0|        0|         1|
|   LHR|     0|              2|        0|         0|
|   MAD|     3|              0|        0|         0|
+------+------+---------------+---------+----------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [origin#27, coalesce(__pivot_count(1) AS count AS `count(1) AS count`#3426[0], 0) AS Iberia#3439L, coalesce(__pivot_count(1) AS count AS `count(1) AS count`#3426[1], 0) AS British Airways#3440L, coalesce(__pivot_count(1) AS count AS `count(1) AS count`#3426[2], 0) AS Lufthansa#3441L, coalesce(__pivot_count(1) AS count AS `count(1) AS count`#3426[3], 0) AS Air France#3442L]
   +- HashAggregate(keys=[origin#27], functions=[pivotfirst(airline#26, count(1) AS count#3416L, Iberia, British Airways, Lufthansa,